In [1]:
import numpy as np
import tifffile as tiff
from skimage.measure import label as label_connected_components
import os

def track_and_relabel_3d_masks(mask_stack_path, output_path):
    print(f"Loading masks from: {mask_stack_path}")
    
    # 1. Load the data
    # We read the raw mask stack from the TIFF
    raw_stack = tiff.imread(mask_stack_path)
    
    # Get dimensions
    num_slices, height, width = raw_stack.shape
    print(f"Volume shape: {num_slices} slices, {height}x{width} pixels")

    # 2. Initialize Result Container
    # We create a brand new empty stack to avoid "ID collisions"
    result_stack = np.zeros_like(raw_stack, dtype=np.uint16)
    
    # This counter ensures every new object gets a globally unique ID
    # We start at 1 because 0 is reserved for background
    next_global_id = 1
    
    # 3. Process the First Slice
    # We run connected components on the first slice to clean up any duplicate labels
    # from the inference model and establish the "Ground Truth" for tracking.
    first_slice_binary = raw_stack[0] > 0
    first_slice_labeled = label_connected_components(first_slice_binary)
    
    # Transfer unique labels to result_stack and update our ID counter
    if first_slice_labeled.max() > 0:
        result_stack[0] = first_slice_labeled
        next_global_id = first_slice_labeled.max() + 1
    
    print(f"Slice 0 processed. Found {next_global_id - 1} objects.")

    # 4. Loop through the remaining slices
    for i in range(1, num_slices):
        
        # A. Pre-process Current Slice (The "Fix")
        # Ensure every distinct blob has a unique ID locally (1, 2, 3...)
        # We ignore the raw labels and just look at the shapes.
        current_slice_binary = raw_stack[i] > 0
        current_slice_temp_labels = label_connected_components(current_slice_binary)
        
        # Get list of objects in this current slice (skipping 0 background)
        current_obj_ids = np.unique(current_slice_temp_labels)
        current_obj_ids = current_obj_ids[current_obj_ids != 0]
        
        # B. Match Objects to Previous Slice
        for obj_id in current_obj_ids:
            # Create a mask for the current object we are trying to identify
            current_obj_mask = (current_slice_temp_labels == obj_id)
            current_obj_size = np.sum(current_obj_mask)
            
            # Look at the *same location* in the PREVIOUS RESULT slice
            # We want to see what IDs are "underneath" this object
            prev_labels_underneath = result_stack[i-1][current_obj_mask]
            
            # Count the frequency of each label found underneath
            # 'counts' will tell us: {Label 5: 100 pixels, Label 0: 20 pixels}
            unique_labels, counts = np.unique(prev_labels_underneath, return_counts=True)
            
            # Filter out background (0) from the potential matches
            valid_mask = unique_labels != 0
            valid_labels = unique_labels[valid_mask]
            valid_counts = counts[valid_mask]
            
            match_found = False
            best_label = None
            
            if len(valid_labels) > 0:
                # Find the label with the maximum overlap
                max_idx = np.argmax(valid_counts)
                max_overlap_count = valid_counts[max_idx]
                candidate_label = valid_labels[max_idx]
                
                # C. Validation Logic (Optional but recommended)
                # We calculate a simplified Intersection over Union (IoU) or Coverage
                # If the overlap is tiny (e.g., just touching), we might treat it as new.
                # Here, we check if >25% of the current object overlaps with the candidate.
                overlap_ratio = max_overlap_count / current_obj_size
                
                if overlap_ratio > 0.25:
                    best_label = candidate_label
                    match_found = True
            
            # D. Assign the Label
            if match_found:
                # It's the same object as before!
                result_stack[i][current_obj_mask] = best_label
            else:
                # It's a new object! Assign a brand new global ID
                result_stack[i][current_obj_mask] = next_global_id
                next_global_id += 1
        
        if i % 10 == 0:
            print(f"Processed slice {i}/{num_slices}...")

    # 5. Save the result
    print(f"Saving tracked masks to: {output_path}")
    tiff.imwrite(output_path, result_stack)
    print("Done!")

# --- USAGE ---
# Replace these paths with your actual file paths
input_tif = r"/home/students/rawat/czi_output/Annotations/PN_KC_488_568_Brain1_CalyxL_LSMPlus_16bit_label.tif"
output_tif = r"/home/students/rawat/czi_output/Annotations/PN_KC_488_568_Brain1_CalyxL_LSMPlus_16bit_label.tif"

# Only run if the file actually exists to prevent errors
if os.path.exists(input_tif):
    track_and_relabel_3d_masks(input_tif, output_tif)
else:
    print(f"Please update the 'input_tif' path. File not found: {input_tif}")

Loading masks from: /home/students/rawat/czi_output/Annotations/PN_KC_488_568_Brain1_CalyxL_LSMPlus_16bit_label.tif
Volume shape: 75 slices, 1101x1101 pixels
Slice 0 processed. Found 1 objects.
Processed slice 10/75...
Processed slice 20/75...
Processed slice 30/75...
Processed slice 40/75...
Processed slice 50/75...
Processed slice 60/75...
Processed slice 70/75...
Saving tracked masks to: /home/students/rawat/czi_output/Annotations/PN_KC_488_568_Brain1_CalyxL_LSMPlus_16bit_label.tif
Done!
